# penalty

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    cohen_kappa_score,
    mean_absolute_error,
)
from scipy.stats import spearmanr

In [ ]:
ALPHA = 0.2

expert_df = pd.read_excel("expert_label.xlsx")
llm_df = pd.read_excel("sample/llms_scored.xlsx")
abstract_df = pd.read_excel("sample/abstract_only_scores.xlsx")

merged_df = (
    expert_df
    .merge(
        llm_df[
            [
                "instance_id",
                "gpt_5_mini_score",
            ]
        ],
        on="instance_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        abstract_df[
            [
                "instance_id",
                "cross_abstract_score",
                "gpt_5_mini_abstract_score",
            ]
        ],
        on="instance_id",
        how="left",
        validate="one_to_one",
    )
)

In [ ]:
def calibrate_lambda(merged_df, show_plot=True):

    alpha = 0.2

    # 专家标签：1、2、3、4
    expert_label = (
        merged_df["final_label"]
        .astype(int)
        .to_numpy()
    )

    # 映射为：0.125、0.375、0.625、0.875
    expert_score = merged_df["final_score"]

    # Abstract-only融合分数
    abstract_score = (
        alpha
        * merged_df["cross_abstract_score"].to_numpy()
        + (1 - alpha)
        * merged_df["gpt_5_mini_abstract_score"].to_numpy()
    )

    rows = []

    for lambda_value in np.arange(0.50, 1.001, 0.01):

        penalized_score = (
            lambda_value * abstract_score
        )

        # 连续分数转换为1、2、3、4
        predicted_label = np.clip(
            np.floor(penalized_score * 4) + 1,
            1,
            4,
        ).astype(int)

        mae = mean_absolute_error(
            expert_score,
            penalized_score,
        )

        qwk = cohen_kappa_score(
            expert_label,
            predicted_label,
            weights="quadratic",
        )

        rows.append(
            {
                "lambda": round(lambda_value, 2),
                "mae": mae,
                "qwk": qwk,
            }
        )

    lambda_df = pd.DataFrame(rows)

    # -------------------------------------------------
    # 同时考虑较低的MAE和较高的QWK
    # -------------------------------------------------

    mae_min = lambda_df["mae"].min()
    mae_max = lambda_df["mae"].max()

    qwk_min = lambda_df["qwk"].min()
    qwk_max = lambda_df["qwk"].max()


    # MAE越小越好
    lambda_df["mae_loss"] = (
        lambda_df["mae"] - mae_min
    ) / (
        mae_max - mae_min + 1e-12
    )


    # QWK越大越好，因此计算距离最高QWK的损失
    lambda_df["qwk_loss"] = (
        qwk_max - lambda_df["qwk"]
    ) / (
        qwk_max - qwk_min + 1e-12
    )


    # 到理想点“最低MAE、最高QWK”的距离
    lambda_df["distance_to_ideal"] = np.sqrt(
        lambda_df["mae_loss"] ** 2
        + lambda_df["qwk_loss"] ** 2
    )


    best_index = (
        lambda_df["distance_to_ideal"]
        .idxmin()
    )

    best_result = lambda_df.loc[
        best_index
    ]

    selected_lambda = float(
        best_result["lambda"]
    )

    lambda_df["selected"] = (
        lambda_df["lambda"]
        == selected_lambda
    )

    # Spearman不随正数lambda变化
    rho = spearmanr(
        expert_score,
        abstract_score,
    ).statistic

    print(f"Selected lambda: {selected_lambda:.2f}")
    print(f"Spearman: {rho:.4f}")
    print(f"MAE: {best_result['mae']:.4f}")
    print(f"QWK: {best_result['qwk']:.4f}")
    print(
        "Distance to ideal:",
        f"{best_result['distance_to_ideal']:.4f}",
    )

    if show_plot:

        fig, ax = plt.subplots(
            figsize=(5, 3)
        )
        

        ax.plot(
            lambda_df["lambda"],
            lambda_df["mae"],
            marker="o",
            markersize=3,
            linewidth=1,
            label=r"MAE ($\downarrow$)"
        )


        ax.plot(
            lambda_df["lambda"],
            lambda_df["qwk"],
            marker="s",
            markersize=3,
            linewidth=1,
            label=r"QWK ($\uparrow$)"
        )

        ax.axvline(
            selected_lambda,
            color="red",
            linestyle="--",
            linewidth=1.5,
            label=(
                r"$\lambda_{\mathrm{cal}}$"
                f"={selected_lambda:.2f}"
            ),
        )


        ax.set_xlabel(
            r"Calibration factor $\lambda_{\mathrm{cal}}$"
        )

        ax.set_ylabel(
            "Metric value"
        )

        ax.set_xlim(
            0.5,
            1.0,
        )

        ax.set_ylim(
            0.0,
            1.0,
        )

        ax.grid(
            linestyle=":",
            alpha=0.6,
        )

        ax.legend(
            frameon=False
        )

        plt.tight_layout()
        plt.show()

    return lambda_df
lambda_df = calibrate_lambda(merged_df, True)

In [ ]:
lambda_df[
    lambda_df["lambda"].isin(
        [0.80, 0.90, 0.95, 1.00]
    )
][
    [
        "lambda",
        "mae",
        "qwk",
        "distance_to_ideal",
    ]
].to_latex()